# DMLF — Cross-domain strict BGL → HDFS

**Basé sur les Colab DMLF leak-safe V3 (LR, SVM, RF, XGBoost) et le pipeline DMLF final.**

Ce notebook réalise une expérience **directionnelle stricte** :

`BGL_TRAIN → modèle BGL → prédiction sur HDFS`

### Pipeline complet
1. Collecte de `HDFS.log`, `anomaly_label.csv/anomaly.csv`, `BGL.log`  
2. Construction des instances : HDFS = sessions `block_id`, BGL = fenêtres 40/20  
3. Split anti-fuite du **source** 80/20 ; pour BGL le split est effectué par blocs bruts avant fenêtrage  
4. Normalisation  
5. Drain3 **fit uniquement sur SOURCE_TRAIN**  
6. Transformation SOURCE_TEST et TARGET avec parser gelé  
7. `sequence_text` + TF-IDF unigrams/bigrams (max 5000) + statistiques partagées  
8. Entraînement séparé de LR, SVM calibré, RF, XGBoost sur SOURCE_TRAIN  
9. Évaluation in-domain sur SOURCE_TEST  
10. Évaluation cross-domain stricte sur **HDFS**  
11. Accuracy, Precision, Recall, F1, AUROC, AUPR, Brier, matrice de confusion, ROC et PR.

> **Important :** il n'y a pas de late-fusion dans une expérience `BGL→HDFS` stricte, car entraîner un `Model_HDFS` supervisé utiliserait les labels du target et transformerait le scénario en multi-system / supervised-target. Ce notebook représente donc le **bras cross-domain de DMLF**.


In [ ]:
# ============================================================
# 0. Installation + imports
# ============================================================
!pip -q install drain3 xgboost

import os, re, random, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from scipy import sparse
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, roc_curve, precision_recall_curve
)
from xgboost import XGBClassifier

from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('Environnement prêt.')


In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================
DIRECTION = "BGL_TO_HDFS"

# Google Drive : mettre False si les fichiers sont déjà dans /content
USE_GOOGLE_DRIVE = True
DRIVE_MOUNT = "/content/drive"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    if not os.path.exists(DRIVE_MOUNT) or not os.listdir(DRIVE_MOUNT):
        drive.mount(DRIVE_MOUNT)

SEARCH_ROOTS = [Path('/content')]
if USE_GOOGLE_DRIVE:
    SEARCH_ROOTS.insert(0, Path('/content/drive/MyDrive'))

# Noms acceptés
HDFS_LOG_CANDIDATES   = ['HDFS.log', 'hdfs.log']
HDFS_LABEL_CANDIDATES = ['anomaly_label.csv', 'anomaly.csv', 'anomaly_labels.csv']
BGL_LOG_CANDIDATES    = ['BGL.log', 'bgl.log']

# Protocole identique aux DMLF leak-safe V3
TEST_SIZE = 0.20
BGL_BLOCK_SIZE = 400
BGL_WINDOW_SIZE = 40
BGL_STRIDE = 20
BGL_SAMPLE_FRAC = 1.0   # 1.0 = toutes les données BGL disponibles

TFIDF_MAX_FEATURES = 5000
TFIDF_NGRAM_RANGE = (1, 2)
TFIDF_MIN_DF = 2
TEXT_SVD_COMPONENTS = 256

# Les quatre familles DMLF testées
FAMILIES = ['lr', 'svm', 'rf', 'xgb']

# IMPORTANT : mode cross-domain strict.
# Le parser et le TF-IDF sont ajustés uniquement sur SOURCE_TRAIN.
TARGET_PARSER_MODE = 'SOURCE_FROZEN'

# Aucun tuning de seuil sur le target : seuil fixe.
DECISION_THRESHOLD = 0.50

# Laisser None pour utiliser toutes les instances target.
# Pour un test rapide, mettre par exemple 20000.
MAX_TARGET_INSTANCES = None

print('Direction :', DIRECTION)
print('Mode parser target :', TARGET_PARSER_MODE)


In [ ]:
# ============================================================
# 2. Recherche des fichiers + chargement robuste
# ============================================================
def find_file(candidates, roots=SEARCH_ROOTS):
    for root in roots:
        if not root.exists():
            continue
        for name in candidates:
            direct = root / name
            if direct.exists():
                return direct
        # recherche récursive, arrêt au premier match
        for name in candidates:
            hits = list(root.rglob(name))
            if hits:
                return hits[0]
    raise FileNotFoundError(f'Fichier introuvable. Candidats={candidates}, roots={roots}')

HDFS_LOG_PATH = find_file(HDFS_LOG_CANDIDATES)
HDFS_LABEL_PATH = find_file(HDFS_LABEL_CANDIDATES)
BGL_LOG_PATH = find_file(BGL_LOG_CANDIDATES)

print('HDFS.log       :', HDFS_LOG_PATH)
print('HDFS labels    :', HDFS_LABEL_PATH)
print('BGL.log        :', BGL_LOG_PATH)

def load_hdfs_labels(label_csv_path):
    df = pd.read_csv(label_csv_path)
    block_col, label_col = None, None
    for c in df.columns:
        cl = c.lower()
        if 'block' in cl:
            block_col = c
        if 'label' in cl:
            label_col = c
    if block_col is None:
        block_col = df.columns[0]
    if label_col is None:
        label_col = df.columns[1]
    out = df[[block_col, label_col]].copy()
    out.columns = ['block_id', 'label_raw']
    out['label'] = out['label_raw'].astype(str).str.lower().map(
        lambda x: 1 if ('anomaly' in x or 'abnormal' in x or x.strip() == '1') else 0
    )
    out['block_id'] = out['block_id'].astype(str).str.strip()
    return out[['block_id','label']]

def extract_block_id(text):
    m = re.search(r'(blk_-?\d+)', str(text))
    return m.group(1) if m else None

def build_hdfs_sessions(hdfs_log_path, hdfs_label_path):
    labels_df = load_hdfs_labels(hdfs_label_path)
    label_map = dict(zip(labels_df['block_id'], labels_df['label']))
    groups = {}
    with open(hdfs_log_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in tqdm(f, desc='Lecture HDFS'):
            line = line.rstrip('\n')
            block_id = extract_block_id(line)
            if block_id is None or block_id not in label_map:
                continue
            groups.setdefault(block_id, []).append(line)
    rows=[]
    for block_id, msgs in groups.items():
        rows.append({
            'sample_id': f'HDFS::{block_id}',
            'dataset': 'HDFS',
            'group_id': block_id,
            'messages': msgs,
            'label': int(label_map[block_id]),
        })
    return pd.DataFrame(rows)

def read_bgl_stream(bgl_log_path):
    lines, labels = [], []
    with open(bgl_log_path, 'r', encoding='utf-8', errors='ignore') as f:
        for raw in tqdm(f, desc='Lecture BGL'):
            raw = raw.rstrip('\n')
            if not raw.strip():
                continue
            first = raw.split()[0]
            label = 0 if first == '-' else 1
            lines.append(raw)
            labels.append(label)
    return lines, labels

def make_bgl_blocks(lines, labels, block_size=400, min_len=40):
    blocks=[]
    for block_id, start in enumerate(range(0, len(lines), block_size)):
        end=min(len(lines), start+block_size)
        blines=lines[start:end]
        blabels=labels[start:end]
        if len(blines) < min_len:
            continue
        blocks.append({
            'block_id': block_id, 'start': start, 'end': end,
            'lines': blines, 'labels': blabels,
            'label': int(max(blabels))
        })
    return blocks

def sample_bgl_blocks(blocks, frac=1.0, seed=42):
    if frac >= 1.0:
        return list(blocks)
    idx=np.arange(len(blocks)); y=np.array([b['label'] for b in blocks], dtype=int)
    n_target=max(2,int(len(blocks)*frac))
    try:
        sampled_idx,_=train_test_split(idx, train_size=n_target, stratify=y, random_state=seed)
        sampled_idx=set(sampled_idx.tolist())
        return [b for i,b in enumerate(blocks) if i in sampled_idx]
    except Exception:
        blocks=list(blocks); random.Random(seed).shuffle(blocks); return blocks[:n_target]

def split_bgl_blocks(blocks, test_size=0.20, seed=42):
    idx=np.arange(len(blocks)); y=np.array([b['label'] for b in blocks],dtype=int)
    try:
        tr_idx,te_idx=train_test_split(idx,test_size=test_size,stratify=y,random_state=seed)
    except Exception:
        tr_idx,te_idx=train_test_split(idx,test_size=test_size,random_state=seed)
    tr=set(tr_idx.tolist()); te=set(te_idx.tolist())
    return [b for i,b in enumerate(blocks) if i in tr], [b for i,b in enumerate(blocks) if i in te]

def build_windows_from_blocks(blocks, window_size=40, stride=20):
    rows=[]
    for block in tqdm(blocks, desc='Fenêtres BGL par blocs'):
        lines=block['lines']; labels=block['labels']
        for local_start in range(0, max(0, len(lines)-window_size+1), stride):
            local_end=local_start+window_size
            win_lines=lines[local_start:local_end]
            win_labels=labels[local_start:local_end]
            gs=block['start']+local_start; ge=block['start']+local_end
            rows.append({
                'sample_id': f"BGL::{block['block_id']}_{gs}_{ge}",
                'dataset':'BGL',
                'group_id':f"{block['block_id']}:{gs}_{ge}",
                'messages':win_lines,
                'label':int(max(win_labels)),
            })
    return pd.DataFrame(rows)

def print_distribution(df, name):
    c=df['label'].value_counts().sort_index().to_dict() if len(df) else {}
    print(f'{name}: shape={df.shape}, labels={c}')


In [ ]:
# ============================================================
# 3. Prétraitement + Drain3 + log sequences
# ============================================================
def normalize_text(text):
    text = str(text).strip()
    text = re.sub(r'0x[0-9a-fA-F]+', ' HEX ', text)
    text = re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', ' IP ', text)
    text = re.sub(r'blk_-?\d+', ' BLOCKID ', text)
    text = re.sub(r'\b\d{5,}\b', ' NUM ', text)
    text = re.sub(r'\b\d+\b', ' NUM ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

def build_template_miner():
    cfg=TemplateMinerConfig()
    return TemplateMiner(config=cfg)

def fit_parse_templates(train_df, text_col='messages'):
    """Drain3 FIT uniquement sur SOURCE_TRAIN."""
    miner=build_template_miner(); templates_out=[]
    for msgs in tqdm(train_df[text_col].tolist(), desc='Drain3 FIT + parse SOURCE_TRAIN'):
        seq=[]
        for msg in msgs:
            msg_norm=normalize_text(msg)
            result=miner.add_log_message(msg_norm)
            tpl=result.get('template_mined') if isinstance(result,dict) else None
            if tpl is None: tpl=msg_norm
            seq.append(tpl)
        templates_out.append(seq)
    out=train_df.copy(); out['templates']=templates_out
    out['sequence_text']=[' [SEP] '.join(x) for x in templates_out]
    out['parser_unmatched_ratio']=0.0
    return out, miner

def transform_parse_templates(df, miner, text_col='messages', desc='Drain3 frozen transform'):
    """Aucun apprentissage Drain3 sur SOURCE_TEST ni TARGET."""
    templates_out=[]; ratios=[]
    for msgs in tqdm(df[text_col].tolist(), desc=desc):
        seq=[]; unmatched=0
        for msg in msgs:
            msg_norm=normalize_text(msg)
            cluster=miner.match(msg_norm)
            if cluster is not None:
                tpl=cluster.get_template()
            else:
                tpl=msg_norm
                unmatched += 1
            seq.append(tpl)
        templates_out.append(seq)
        ratios.append(unmatched/max(1,len(msgs)))
    out=df.copy(); out['templates']=templates_out
    out['sequence_text']=[' [SEP] '.join(x) for x in templates_out]
    out['parser_unmatched_ratio']=ratios
    return out

ANOMALY_TERMS = [
    'error','exception','fatal','fail','failed','warn','warning',
    'abort','panic','timeout','timed out','unable','refused'
]
ANOMALY_PATTERN = re.compile('|'.join(re.escape(x) for x in ANOMALY_TERMS), re.IGNORECASE)

def add_sequence_features(df):
    df=df.copy()
    df['seq_len_raw']=df['messages'].apply(len)
    df['n_templates']=df['templates'].apply(len)
    df['n_unique_templates']=df['templates'].apply(lambda x: len(set(x)))
    df['anomaly_keyword_count']=df['sequence_text'].apply(lambda x: len(ANOMALY_PATTERN.findall(x)))
    df['anomaly_keyword_ratio']=df['anomaly_keyword_count']/df['n_templates'].clip(lower=1)
    df['avg_template_len']=df['templates'].apply(
        lambda seq: float(np.mean([len(t.split()) for t in seq])) if len(seq) else 0.0
    )
    return df

STAT_COLS = [
    'seq_len_raw','n_templates','n_unique_templates',
    'anomaly_keyword_count','anomaly_keyword_ratio','avg_template_len'
]


In [ ]:
# ============================================================
# 4. Représentation TF-IDF + modèles DMLF
# ============================================================
class FeatureBuilder:
    def __init__(self, model_family='lr'):
        self.model_family=model_family
        self.vectorizer=TfidfVectorizer(
            max_features=TFIDF_MAX_FEATURES,
            ngram_range=TFIDF_NGRAM_RANGE,
            min_df=TFIDF_MIN_DF,
            lowercase=False,
        )
        self.scaler=StandardScaler()
        self.svd=None

    def fit_transform(self, df):
        # FIT exclusivement SOURCE_TRAIN
        X_text=self.vectorizer.fit_transform(df['sequence_text'].tolist())
        X_stats=self.scaler.fit_transform(df[STAT_COLS].values)
        if self.model_family in {'rf','xgb'}:
            n_comp=min(TEXT_SVD_COMPONENTS, max(2, X_text.shape[1]-1))
            self.svd=TruncatedSVD(n_components=n_comp, random_state=SEED)
            X_text=self.svd.fit_transform(X_text)
            return np.hstack([X_text,X_stats])
        return sparse.hstack([X_text,sparse.csr_matrix(X_stats)], format='csr')

    def transform(self, df):
        X_text=self.vectorizer.transform(df['sequence_text'].tolist())
        X_stats=self.scaler.transform(df[STAT_COLS].values)
        if self.model_family in {'rf','xgb'}:
            X_text=self.svd.transform(X_text)
            return np.hstack([X_text,X_stats])
        return sparse.hstack([X_text,sparse.csr_matrix(X_stats)], format='csr')

def compute_scale_pos_weight(y):
    y=np.asarray(y,dtype=int); n_pos=max(1,int((y==1).sum())); n_neg=max(1,int((y==0).sum()))
    return n_neg/n_pos

def make_model(family, y_train):
    family=family.lower()
    if family=='lr':
        return LogisticRegression(max_iter=3000, solver='liblinear', random_state=SEED, class_weight='balanced')
    if family=='svm':
        base=LinearSVC(C=1.0, random_state=SEED, class_weight='balanced')
        return CalibratedClassifierCV(base, method='sigmoid', cv=3)
    if family=='rf':
        return RandomForestClassifier(
            n_estimators=350, max_depth=None, min_samples_split=2, min_samples_leaf=1,
            n_jobs=-1, random_state=SEED, class_weight='balanced_subsample'
        )
    if family=='xgb':
        spw=compute_scale_pos_weight(y_train)
        return XGBClassifier(
            n_estimators=350, max_depth=6, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            objective='binary:logistic', eval_metric='logloss',
            random_state=SEED, n_jobs=-1, scale_pos_weight=spw,
        )
    raise ValueError(f'Famille inconnue: {family}')

def predict_proba_1(model, X):
    if hasattr(model,'predict_proba'):
        return model.predict_proba(X)[:,1]
    scores=np.asarray(model.decision_function(X),dtype=float)
    return 1/(1+np.exp(-scores))

def metric_row(name,y_true,p,threshold=0.5):
    y_true=np.asarray(y_true,dtype=int); p=np.asarray(p,dtype=float)
    y_pred=(p>=threshold).astype(int)
    def safe_auc(fn):
        try: return float(fn(y_true,p))
        except Exception: return np.nan
    return {
        'model':name,
        'accuracy':accuracy_score(y_true,y_pred),
        'precision':precision_score(y_true,y_pred,zero_division=0),
        'recall':recall_score(y_true,y_pred,zero_division=0),
        'f1':f1_score(y_true,y_pred,zero_division=0),
        'auc_roc':safe_auc(roc_auc_score),
        'auc_pr':safe_auc(average_precision_score),
        'brier':brier_score_loss(y_true,p),
    }

def plot_confusion(y_true,p,title,threshold=0.5):
    pred=(np.asarray(p)>=threshold).astype(int); cm=confusion_matrix(y_true,pred)
    plt.figure(figsize=(4,3.4)); plt.imshow(cm); plt.colorbar()
    plt.xticks([0,1],['Normal','Anomaly']); plt.yticks([0,1],['Normal','Anomaly'])
    for i in range(2):
        for j in range(2): plt.text(j,i,str(cm[i,j]),ha='center',va='center')
    plt.title(title); plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# 5. Construction SOURCE / TARGET selon la direction
# ============================================================
print('\n=== Construction des instances ===')
hdfs_df = build_hdfs_sessions(HDFS_LOG_PATH, HDFS_LABEL_PATH)
print_distribution(hdfs_df, 'HDFS complet')

bgl_lines, bgl_labels = read_bgl_stream(BGL_LOG_PATH)
bgl_blocks = make_bgl_blocks(bgl_lines, bgl_labels, block_size=BGL_BLOCK_SIZE, min_len=BGL_WINDOW_SIZE)
bgl_blocks = sample_bgl_blocks(bgl_blocks, frac=BGL_SAMPLE_FRAC, seed=SEED)
print('BGL blocs :', len(bgl_blocks))

if DIRECTION == 'HDFS_TO_BGL':
    # SOURCE HDFS : split sessions 80/20, labels source autorisés
    source_train_raw, source_test_raw = train_test_split(
        hdfs_df, test_size=TEST_SIZE, random_state=SEED, stratify=hdfs_df['label']
    )
    source_train_raw=source_train_raw.reset_index(drop=True)
    source_test_raw=source_test_raw.reset_index(drop=True)

    # TARGET BGL : aucune partie n'est utilisée pour FIT. Toute la cible sert uniquement à l'évaluation.
    target_raw = build_windows_from_blocks(
        bgl_blocks, window_size=BGL_WINDOW_SIZE, stride=BGL_STRIDE
    ).reset_index(drop=True)
    SOURCE_NAME, TARGET_NAME = 'HDFS','BGL'

elif DIRECTION == 'BGL_TO_HDFS':
    # SOURCE BGL : split blocs bruts AVANT fenêtrage => anti-overlap leakage
    src_train_blocks, src_test_blocks = split_bgl_blocks(bgl_blocks, test_size=TEST_SIZE, seed=SEED)
    source_train_raw = build_windows_from_blocks(src_train_blocks, BGL_WINDOW_SIZE, BGL_STRIDE).reset_index(drop=True)
    source_test_raw  = build_windows_from_blocks(src_test_blocks,  BGL_WINDOW_SIZE, BGL_STRIDE).reset_index(drop=True)

    # TARGET HDFS : aucune session n'est utilisée pour FIT.
    target_raw = hdfs_df.reset_index(drop=True)
    SOURCE_NAME, TARGET_NAME = 'BGL','HDFS'
else:
    raise ValueError(DIRECTION)

if MAX_TARGET_INSTANCES is not None and len(target_raw) > MAX_TARGET_INSTANCES:
    # échantillonnage uniquement pour accélérer, jamais pour entraînement
    target_raw = target_raw.sample(MAX_TARGET_INSTANCES, random_state=SEED).reset_index(drop=True)

print_distribution(source_train_raw, f'{SOURCE_NAME} SOURCE_TRAIN')
print_distribution(source_test_raw,  f'{SOURCE_NAME} SOURCE_TEST')
print_distribution(target_raw,       f'{TARGET_NAME} TARGET_EVAL')
print('\nAUDIT: aucune instance TARGET n\'entre dans fit Drain3/TF-IDF/classifieur.')


In [ ]:
# ============================================================
# 6. Parsing strict : FIT sur SOURCE_TRAIN seulement
# ============================================================
source_train, source_miner = fit_parse_templates(source_train_raw)
source_test = transform_parse_templates(
    source_test_raw, source_miner, desc='Drain3 frozen SOURCE_TEST'
)
target_eval = transform_parse_templates(
    target_raw, source_miner, desc=f'Drain3 SOURCE_FROZEN -> {TARGET_NAME}'
)

source_train=add_sequence_features(source_train)
source_test=add_sequence_features(source_test)
target_eval=add_sequence_features(target_eval)

print('\nParser unmatched ratio moyen:')
print(' SOURCE_TEST :', round(float(source_test['parser_unmatched_ratio'].mean()),4))
print(' TARGET_EVAL :', round(float(target_eval['parser_unmatched_ratio'].mean()),4))


In [ ]:
# ============================================================
# 7. Entraînement LR / SVM / RF / XGB sur SOURCE seulement
#    puis prédiction in-domain et cross-domain TARGET
# ============================================================
results=[]
probabilities={}

y_src_train=source_train['label'].values.astype(int)
y_src_test=source_test['label'].values.astype(int)
y_target=target_eval['label'].values.astype(int)  # utilisé UNIQUEMENT ici pour les métriques

for family in FAMILIES:
    print(f'\n================ {family.upper()} ================')
    fb=FeatureBuilder(family)
    X_train=fb.fit_transform(source_train)
    X_source_test=fb.transform(source_test)
    X_target=fb.transform(target_eval)

    model=make_model(family,y_src_train)
    model.fit(X_train,y_src_train)

    p_src=predict_proba_1(model,X_source_test)
    p_tgt=predict_proba_1(model,X_target)

    results.append(metric_row(f'{family.upper()}_{SOURCE_NAME}_on_{SOURCE_NAME}',y_src_test,p_src,DECISION_THRESHOLD))
    results.append(metric_row(f'{family.upper()}_{SOURCE_NAME}_to_{TARGET_NAME}',y_target,p_tgt,DECISION_THRESHOLD))
    probabilities[family]={'source':p_src,'target':p_tgt}

results_df=pd.DataFrame(results)
pd.set_option('display.max_columns',None)
print('\n===== Tableau des métriques =====')
display(results_df.sort_values(['model']).reset_index(drop=True))


In [ ]:
# ============================================================
# 8. Résumé cross-domain + figures + sauvegarde
# ============================================================
cross_df = results_df[results_df['model'].str.contains(f'_to_{TARGET_NAME}')].copy()
print('\n===== Résultats cross-domain uniquement =====')
display(cross_df.sort_values('f1',ascending=False).reset_index(drop=True))

plt.figure(figsize=(8,4))
plt.bar(cross_df['model'],cross_df['f1'])
plt.ylabel('F1'); plt.title(f'DMLF strict cross-domain: {SOURCE_NAME} -> {TARGET_NAME}')
plt.xticks(rotation=25,ha='right'); plt.ylim(0,1); plt.tight_layout(); plt.show()

# Meilleur modèle cross-domain selon F1
best_row=cross_df.sort_values('f1',ascending=False).iloc[0]
best_family=best_row['model'].split('_')[0].lower()
print('Meilleure famille TARGET selon F1 :', best_family.upper())
plot_confusion(y_target,probabilities[best_family]['target'],f'{best_family.upper()} — {SOURCE_NAME}->{TARGET_NAME}')

# Courbes ROC/PR des 4 familles sur target
plt.figure(figsize=(6,5))
for fam in FAMILIES:
    p=probabilities[fam]['target']
    try:
        fpr,tpr,_=roc_curve(y_target,p); plt.plot(fpr,tpr,label=fam.upper())
    except Exception: pass
plt.plot([0,1],[0,1],'--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title(f'ROC — {SOURCE_NAME}->{TARGET_NAME}'); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(6,5))
for fam in FAMILIES:
    p=probabilities[fam]['target']
    try:
        pr,rc,_=precision_recall_curve(y_target,p); plt.plot(rc,pr,label=fam.upper())
    except Exception: pass
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title(f'PR — {SOURCE_NAME}->{TARGET_NAME}'); plt.legend(); plt.tight_layout(); plt.show()

out_csv=f'dmlf_{SOURCE_NAME.lower()}_to_{TARGET_NAME.lower()}_strict_results.csv'
results_df.to_csv(out_csv,index=False)
print('Résultats sauvegardés :',out_csv)

print('\n=== AUDIT FINAL ANTI-FUITE ===')
print('1. Split SOURCE effectué avant tout fit.')
print('2. BGL source: split par blocs bruts avant fenêtres.')
print('3. Drain3 fit = SOURCE_TRAIN uniquement.')
print('4. TF-IDF / StandardScaler / SVD fit = SOURCE_TRAIN uniquement.')
print('5. Classifier fit = SOURCE_TRAIN uniquement.')
print('6. TARGET labels jamais utilisées pour fit/tuning; seulement pour métriques finales.')
print('7. Seuil fixe =',DECISION_THRESHOLD,'; aucun tuning sur TARGET.')
